In [ ]:
import json
import os
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import constants

In [ ]:
INFO_URL = "https://api.isthereanydeal.com/games/info/v2"
CACHE_DIR = "../../data/raw/itad/game_info"
OUTPUT_PATH = "../../data/interim/game_info.csv"

game_ids = (
    pd.read_parquet("../../data/interim/game_list.parquet")["itad_uuid"]
    .dropna()
    .unique()
    .tolist()
)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
def retry_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    return session


def fetch_info(game_id, session):
    cache_path = os.path.join(CACHE_DIR, f"{game_id}.json")
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as file:
            return json.load(file), True

    response = session.get(
        INFO_URL,
        params={"key": constants.API_KEY, "id": game_id},
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    with open(cache_path, "w", encoding="utf-8") as file:
        json.dump(data, file)
    return data, False

In [ ]:
def collect_info(game_ids, rpm=60):
    session = retry_session()
    rows = []

    for index, game_id in enumerate(game_ids, start=1):
        data, cached = fetch_info(game_id, session)
        reviews = data.get("reviews") or []
        steam_review = next(
            (review for review in reviews if review.get("source") == "Steam"),
            {},
        )
        stats = data.get("stats") or {}
        rows.append({
            "itad_uuid": game_id,
            "type": data.get("type"),
            "achievements": data.get("achievements"),
            "mature": data.get("mature"),
            "release_date": data.get("releaseDate"),
            "rank": stats.get("rank"),
            "collected": stats.get("collected"),
            "steam_score": steam_review.get("score"),
            "steam_review_count": steam_review.get("count"),
            "tags": data.get("tags") or [],
        })
        if not cached:
            time.sleep(60 / rpm)
        if index % 500 == 0:
            print(f"Processed {index:,} of {len(game_ids):,} games")

    return pd.DataFrame(rows).sort_values("itad_uuid").reset_index(drop=True)

In [ ]:
game_info = collect_info(game_ids)
game_info.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {len(game_info):,} rows to {OUTPUT_PATH}")